# RetainIQ — Phase 1.2: Data Quality Audit

## Objective

Assess the technical quality of the raw dataset without altering it.

The audit covers:

- Missing values
- Duplicate records and keys
- Numeric integrity
- Financial-value integrity
- Business-range checks
- Leading/trailing whitespace
- Empty strings


## 1. Environment Setup and Data Ingestion


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Project convention:
# RetainIQ/
# ├── data/
# │   └── raw/
# │       └── telco.csv
# └── phase_01_data_audit/
#     └── notebooks/
#
# From this notebook, the raw dataset is two levels up from the phase folder.
DATA_PATH = Path("C:\\RetainIQ — AI-Powered Telecom Customer Retention Intelligence Platform\\telco.csv")

# Fallback for running the notebook alongside the uploaded file during development.
if not DATA_PATH.exists():
    DATA_PATH = Path("telco.csv")

df = pd.read_csv(DATA_PATH)

print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]:,} columns")

Loaded: 7,043 rows × 50 columns


## 2. Exact Duplicate-Row Audit

Exact duplicate rows can indicate repeated ingestion, accidental concatenation, or source-system
duplication. They are checked separately from duplicate customer IDs because the two conditions
answer different questions.

In [3]:
duplicate_rows = int(df.duplicated().sum())

print(f"Exact duplicate rows: {duplicate_rows:,}")

Exact duplicate rows: 0


In [4]:
assert duplicate_rows == 0, "Exact duplicate rows detected."

print("PASS — No exact duplicate rows detected.")

PASS — No exact duplicate rows detected.


## 3. Missing-Value Profile

Missingness is assessed at both count and percentage level.

The percentage is important because a field with a large absolute null count may still represent
a small share of the population, while a seemingly small count can be material in a narrow field.

In [5]:
missing = df.isna().sum().rename("null_count").to_frame()
missing["null_pct"] = (missing["null_count"] / len(df) * 100).round(2)

missing = (
    missing[missing["null_count"] > 0]
    .sort_values("null_count", ascending=False)
)

missing

,null_count,null_pct
Churn Reason,5174,73.46
Churn Category,5174,73.46
Offer,3877,55.05
Internet Type,1526,21.67


### Interpretation

The raw dataset contains missing values in four fields:

- `Churn Category`
- `Churn Reason`
- `Offer`
- `Internet Type`

This is not yet a reason to impute or drop records. Their business meaning is investigated in
the next notebook.

## 4. Null-Key and Uniqueness Validation

A missing or duplicated business key would undermine the customer-level grain and could create
incorrect joins or double-counted revenue later in the project.

In [6]:
key_quality = pd.Series({
    "Missing Customer IDs": int(df["Customer ID"].isna().sum()),
    "Duplicate Customer IDs": int(df["Customer ID"].duplicated().sum()),
    "Unique Customer IDs": int(df["Customer ID"].nunique(dropna=True))
})

key_quality

Missing Customer IDs         0
Duplicate Customer IDs       0
Unique Customer IDs       7043
dtype: int64

In [7]:
assert df["Customer ID"].notna().all()
assert df["Customer ID"].is_unique

print("PASS — Customer ID is complete and unique.")

PASS — Customer ID is complete and unique.


## 5. Financial Integrity Audit

Financial fields are checked for missing values and impossible negative values.

In [8]:
financial_cols = [
    "Monthly Charge",
    "Total Charges",
    "Total Refunds",
    "Total Extra Data Charges",
    "Total Long Distance Charges",
    "Total Revenue",
    "CLTV"
]

financial_audit = pd.DataFrame({
    "Minimum": df[financial_cols].min(),
    "Maximum": df[financial_cols].max(),
    "Missing": df[financial_cols].isna().sum(),
    "Negative Values": (df[financial_cols] < 0).sum()
})

financial_audit

,Minimum,Maximum,Missing,Negative Values
Monthly Charge,18.25,118.75,0,0
Total Charges,18.80,"8,684.80",0,0
Total Refunds,0.00,49.79,0,0
Total Extra Data Charges,0.00,150.00,0,0
Total Long Distance Charges,0.00,"3,564.72",0,0
Total Revenue,21.36,"11,979.34",0,0
CLTV,"2,003.00","6,500.00",0,0


In [9]:
assert df[financial_cols].notna().all().all()
assert (df[financial_cols] >= 0).all().all()

print("PASS — No missing or negative values detected in audited financial fields.")

PASS — No missing or negative values detected in audited financial fields.


## 6. Business-Range Plausibility Checks

Core numeric fields are reviewed against simple domain ranges.

These are **plausibility checks, not business certification**. A value inside a range is not
automatically guaranteed to be correct.

In [10]:
range_audit = pd.DataFrame({
    "Field": [
        "Age",
        "Tenure in Months",
        "Satisfaction Score",
        "Churn Score",
        "CLTV"
    ],
    "Minimum": [
        df["Age"].min(),
        df["Tenure in Months"].min(),
        df["Satisfaction Score"].min(),
        df["Churn Score"].min(),
        df["CLTV"].min()
    ],
    "Maximum": [
        df["Age"].max(),
        df["Tenure in Months"].max(),
        df["Satisfaction Score"].max(),
        df["Churn Score"].max(),
        df["CLTV"].max()
    ]
})

range_audit

,Field,Minimum,Maximum
0,Age,19,80
1,Tenure in Months,1,72
2,Satisfaction Score,1,5
3,Churn Score,5,96
4,CLTV,2003,6500


Observed ranges are:

- Age: **19–80**
- Tenure: **1–72 months**
- Satisfaction Score: **1–5**
- Churn Score: **5–96**
- CLTV: **2,003–6,500**

No obvious range violations are identified by these basic checks.

## 7. Text-Quality Audit

String fields are checked for leading/trailing whitespace. Such inconsistencies can create
hard-to-detect duplicate categories, broken filters, or failed joins.

In [11]:
string_cols = df.select_dtypes(include="object").columns

whitespace_audit = {}
for col in string_cols:
    mask = df[col].notna() & (df[col] != df[col].str.strip())
    whitespace_audit[col] = int(mask.sum())

whitespace_audit = pd.Series(whitespace_audit, name="whitespace_issues").sort_values(ascending=False)

whitespace_audit[whitespace_audit > 0]

Series([], Name: whitespace_issues, dtype: int64)

In [12]:
total_whitespace_issues = int(whitespace_audit.sum())

print(f"Total leading/trailing whitespace issues: {total_whitespace_issues:,}")

Total leading/trailing whitespace issues: 0


### Interpretation

The supplied `telco.csv` contains **0 detected leading/trailing whitespace issues**.

Phase 2 will still defensively strip string values so the cleaning pipeline remains robust to
future source-file variations.

## 8. Empty-String Audit

Empty strings are different from database-style nulls. Both should be checked explicitly.

In [13]:
empty_string_counts = {
    col: int((df[col].fillna("") == "").sum())
    for col in string_cols
}

empty_string_counts = pd.Series(empty_string_counts, name="empty_strings").sort_values(ascending=False)

empty_string_counts[empty_string_counts > 0]

Churn Reason      5174
Churn Category    5174
Offer             3877
Internet Type     1526
Name: empty_strings, dtype: int64

## 9. Notebook 2 Conclusion

The technical audit identifies no exact duplicate rows, no duplicate or missing customer keys,
no missing values in the audited financial fields, no negative financial values, and no detected
leading/trailing whitespace.

The remaining quality questions are primarily **semantic**: what the nulls mean, which records
belong in the historical churn population, and how business fields should be represented.

**Next notebook:** `03_business_field_audit.ipynb`